In [10]:
!pip install -U torch
!pip install -U scipy
!pip install -U scikit-learn
!pip install -U hdbscan
!pip install -U umap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 31.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Purpose of Notebook

The purpose of this notebook is to try clustering with DBSCAN and on the Graph embeddings and see differences

# Loading Data

In [3]:
from pathlib import Path
DATA_FOLDER = Path('/content/drive/MyDrive/DSCI599/data')

In [4]:
import pandas as pd

csv_files = list(DATA_FOLDER.glob("*.csv"))
# currently using one
df = pd.read_csv(csv_files[0])
df.head()

,int_source_ip,destination_port,protocol,num_scanned_destination_ips,num_unique_flows,num_packets,same_packet_size_flag,avg_packet_size,num_unique_Bs_scanned,num_unique_Cs_scanned,num_unique_Ds_scanned,num_scanned_24_blocks,num_non_conficker_destinations,time_activity,ip_scanned_rate
0,16968188,23,6,316.0,316.0,945.0,1.0,60.0,175.0,182.0,177.0,316.0,65.0,3540.0,0.089266
1,17372742,445,6,661.0,661.0,1322.0,1.0,48.0,125.0,234.0,126.0,654.0,661.0,3540.0,0.186723
2,17384594,445,6,169.0,169.0,329.0,1.0,48.0,98.0,117.0,89.0,169.0,169.0,3360.0,0.050298
3,17400985,23,6,256.0,256.0,762.0,1.0,60.0,1.0,1.0,256.0,1.0,128.0,0.0,-1.000000
4,17401572,445,6,571.0,571.0,1142.0,1.0,48.0,124.0,232.0,125.0,564.0,571.0,3540.0,0.161299


# Load Graph Embeddings

In [5]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [6]:
import torch

source_ip_embeddings = torch.load(DATA_FOLDER / "source_ip_embeddings.pt", map_location=torch.device(device))

In [7]:
source_ip_embeddings.shape

torch.Size([132956, 16])

In [8]:
if device == 'cpu':
  source_ip_embeddings = source_ip_embeddings.detach().numpy()
else:
  source_ip_embeddings = source_ip_embeddings.cpu().detach().numpy()

# Clustering on Graph Embeddings

In [41]:
import scipy.sparse.linalg
import hdbscan
import numpy as np

# graph_clusters = DBSCAN(eps=0.5,
#                         min_samples=5,
#                         algorithm='kd_tree',
#                         n_jobs=-1).fit(source_ip_embeddings)
graph_clusters = hdbscan.HDBSCAN(
    min_cluster_size=50,
    min_samples=25,
    metric='euclidean',
    cluster_selection_method='eom'
)

graph_clusters.fit(source_ip_embeddings)

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


HDBSCAN(min_cluster_size=50, min_samples=25)

In [44]:
graph_clusters.labels_

array([16, 15, -1, ..., -1, -1, 16])

In [46]:
np.unique(graph_clusters.labels_, return_counts=True)

(array([-1,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,
        16, 17]),
 array([13600,    51,    82,   344,  2128,    85,    69,    51,   501,
          184,   251,   114,   114,   642,  3233,  2536, 11676, 80470,
        16825]))

In [47]:
df['graph_labels'] = graph_clusters.labels_
df['graph_labels'].to_csv(DATA_FOLDER / "graph_labels.csv", index=False)

# Clustering

PCA because it keeps crashing my RAM
So lower the dimension
nevermind!

In [48]:
X = df.drop(columns = ['int_source_ip', 'destination_port', 'protocol'])
len(X.columns)

14

In [49]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from sklearn.decomposition import PCA

# there is a problem with infinity, will fix later!
X.loc[X['avg_packet_size'].isin([np.inf, -np.inf]), 'avg_packet_size'] = -1
# for col in X.columns:
#   X.loc[:, col] = MinMaxScaler().fit_transform(X.loc[:, col])
# as like paper [2], minMaxScale X to normalize it
X = MinMaxScaler().fit_transform(X)
# pca = PCA(n_components=8)
# X_pca = pca.fit_transform(X)

In [50]:
from sklearn.cluster import DBSCAN

normal_clusters = hdbscan.HDBSCAN(
    min_cluster_size=500,
    min_samples=250,
    metric='euclidean',
    cluster_selection_method='eom'
)
normal_clusters.fit(X)

normal_clusters.labels_
np.unique(normal_clusters.labels_, return_counts=True)

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


(array([-1,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,
        16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26]),
 array([46571,  3578,  3132,   528,  1313,   654,   554,   712,  1671,
          784,   840,  8370,  1438,   781, 42329,  1687,  1509,  1029,
          924,  1020,   862,   654,  6206,  1189,   831,   592,  1329,
         1869]))

# Save the labels to the dataframe because it takes so long to run

In [53]:
df['normal_labels'] = normal_clusters.labels_
df['normal_labels'].to_csv(DATA_FOLDER / "normal_labels.csv", index=False)

In [55]:
df.loc[df['avg_packet_size'].isin([np.inf, -np.inf]), 'avg_packet_size'] = -1

df.to_csv(DATA_FOLDER / "df_cluster.csv", index=False)